In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Read claims data from Silver layer
df = spark.table("primeinsurance.silver_layer.silver_claims")

display(df)

claim_id,claim_logged_on,_raw_logged_on,claim_logged_on_corrupted,claim_processed_on,_raw_processed_on,claim_processed_on_corrupted,claim_rejected,policy_id,authorities_contacted,bodily_injuries,collision_type,incident_city,incident_date,_raw_incident_date,incident_date_corrupted,incident_location,incident_severity,incident_state,incident_type,injury,number_of_vehicles_involved,police_report_available,property,property_damage,vehicle,witnesses,source_file,ingestion_timestamp
1367,null,34:00.0,true,null,NULL,false,Y,174701,Other,2,Rear Collision,Columbus,null,27:00.0,true,1809 Sky St,Minor Damage,PA,Single Vehicle Collision,8500.00,1,null,2950.00,null,1050.00,3,claims_1.json,2026-03-26T08:28:23.314Z
1430,null,03:00.0,true,null,04:00.0,true,N,322613,Ambulance,2,Rear Collision,Columbus,null,45:00.0,true,3492 Britain St,Major Damage,VA,Single Vehicle Collision,6000.00,1,NO,850.00,null,12000.00,3,claims_1.json,2026-03-26T08:28:23.314Z
1804,null,45:00.0,true,null,54:00.0,true,N,984456,Fire,0,Front Collision,Hillsdale,null,18:00.0,true,5839 Weaver Lane,Major Damage,WV,Multi-vehicle Collision,1300.00,3,YES,130.00,null,6400.00,2,claims_1.json,2026-03-26T08:28:23.314Z
1820,null,34:00.0,true,null,NULL,false,Y,327488,Ambulance,0,Front Collision,Arlington,null,44:00.0,true,1806 Weaver Ridge,Minor Damage,WV,Single Vehicle Collision,6500.00,1,null,425.00,null,3800.00,3,claims_1.json,2026-03-26T08:28:23.314Z
2198,null,47:00.0,true,null,06:00.0,true,N,488037,None,0,null,Columbus,null,10:00.0,true,6888 Elm Ridge,Minor Damage,NY,Vehicle Theft,4750.00,1,YES,415.00,NO,7000.00,3,claims_1.json,2026-03-26T08:28:23.314Z
2565,null,46:00.0,true,null,NULL,false,Y,987905,Police,0,Rear Collision,Hillsdale,null,58:00.0,true,4242 Rock Lane,Major Damage,NC,Single Vehicle Collision,950.00,1,null,500.00,null,6500.00,2,claims_1.json,2026-03-26T08:28:23.314Z
3863,null,59:00.0,true,null,NULL,false,Y,444035,Ambulance,0,Rear Collision,Arlington,null,38:00.0,true,7930 Texas Ave,Minor Damage,SC,Multi-vehicle Collision,6500.00,4,NO,55.00,NO,3000.00,0,claims_1.json,2026-03-26T08:28:23.314Z
3866,null,33:00.0,true,null,NULL,false,N,395269,Other,2,Rear Collision,Hillsdale,null,10:00.0,true,7405 Oak St,Minor Damage,VA,Multi-vehicle Collision,8110.00,3,NO,350.00,null,3500.00,0,claims_1.json,2026-03-26T08:28:23.314Z
4931,null,52:00.0,true,null,04:00.0,true,N,804608,Fire,0,Side Collision,Riverwood,null,35:00.0,true,1472 4th Drive,Minor Damage,WV,Single Vehicle Collision,2800.00,1,YES,420.00,YES,7000.00,1,claims_1.json,2026-03-26T08:28:23.314Z
5005,null,42:00.0,true,null,NULL,false,N,783494,Other,1,Side Collision,Hillsdale,null,26:00.0,true,3447 Solo Ave,Minor Damage,WV,Single Vehicle Collision,2000.00,1,NO,625.00,NO,6000.00,0,claims_1.json,2026-03-26T08:28:23.314Z


In [0]:
# Clean numeric columns (replace ?, NULL with 0)
numeric_cols = ["injury", "property", "vehicle"]


# Create total claim amount
df = df.withColumn(
    "total_amount",
    F.col("injury") + F.col("property") + F.col("vehicle")
)

# Convert dates
df = df.withColumn("Claim_Logged_On", F.to_date("Claim_Logged_On")) \
       .withColumn("Claim_Processed_On", F.to_date("Claim_Processed_On"))

# Processing delay
df = df.withColumn(
    "processing_delay",
    F.datediff("Claim_Processed_On", "Claim_Logged_On")
)

display(df)

claim_id,Claim_Logged_On,_raw_logged_on,claim_logged_on_corrupted,Claim_Processed_On,_raw_processed_on,claim_processed_on_corrupted,claim_rejected,policy_id,authorities_contacted,bodily_injuries,collision_type,incident_city,incident_date,_raw_incident_date,incident_date_corrupted,incident_location,incident_severity,incident_state,incident_type,injury,number_of_vehicles_involved,police_report_available,property,property_damage,vehicle,witnesses,source_file,ingestion_timestamp,total_amount,processing_delay
1367,null,34:00.0,true,null,NULL,false,Y,174701,Other,2,Rear Collision,Columbus,null,27:00.0,true,1809 Sky St,Minor Damage,PA,Single Vehicle Collision,8500.00,1,null,2950.00,null,1050.00,3,claims_1.json,2026-03-26T08:28:23.314Z,12500.00,null
1430,null,03:00.0,true,null,04:00.0,true,N,322613,Ambulance,2,Rear Collision,Columbus,null,45:00.0,true,3492 Britain St,Major Damage,VA,Single Vehicle Collision,6000.00,1,NO,850.00,null,12000.00,3,claims_1.json,2026-03-26T08:28:23.314Z,18850.00,null
1804,null,45:00.0,true,null,54:00.0,true,N,984456,Fire,0,Front Collision,Hillsdale,null,18:00.0,true,5839 Weaver Lane,Major Damage,WV,Multi-vehicle Collision,1300.00,3,YES,130.00,null,6400.00,2,claims_1.json,2026-03-26T08:28:23.314Z,7830.00,null
1820,null,34:00.0,true,null,NULL,false,Y,327488,Ambulance,0,Front Collision,Arlington,null,44:00.0,true,1806 Weaver Ridge,Minor Damage,WV,Single Vehicle Collision,6500.00,1,null,425.00,null,3800.00,3,claims_1.json,2026-03-26T08:28:23.314Z,10725.00,null
2198,null,47:00.0,true,null,06:00.0,true,N,488037,None,0,null,Columbus,null,10:00.0,true,6888 Elm Ridge,Minor Damage,NY,Vehicle Theft,4750.00,1,YES,415.00,NO,7000.00,3,claims_1.json,2026-03-26T08:28:23.314Z,12165.00,null
2565,null,46:00.0,true,null,NULL,false,Y,987905,Police,0,Rear Collision,Hillsdale,null,58:00.0,true,4242 Rock Lane,Major Damage,NC,Single Vehicle Collision,950.00,1,null,500.00,null,6500.00,2,claims_1.json,2026-03-26T08:28:23.314Z,7950.00,null
3863,null,59:00.0,true,null,NULL,false,Y,444035,Ambulance,0,Rear Collision,Arlington,null,38:00.0,true,7930 Texas Ave,Minor Damage,SC,Multi-vehicle Collision,6500.00,4,NO,55.00,NO,3000.00,0,claims_1.json,2026-03-26T08:28:23.314Z,9555.00,null
3866,null,33:00.0,true,null,NULL,false,N,395269,Other,2,Rear Collision,Hillsdale,null,10:00.0,true,7405 Oak St,Minor Damage,VA,Multi-vehicle Collision,8110.00,3,NO,350.00,null,3500.00,0,claims_1.json,2026-03-26T08:28:23.314Z,11960.00,null
4931,null,52:00.0,true,null,04:00.0,true,N,804608,Fire,0,Side Collision,Riverwood,null,35:00.0,true,1472 4th Drive,Minor Damage,WV,Single Vehicle Collision,2800.00,1,YES,420.00,YES,7000.00,1,claims_1.json,2026-03-26T08:28:23.314Z,10220.00,null
5005,null,42:00.0,true,null,NULL,false,N,783494,Other,1,Side Collision,Hillsdale,null,26:00.0,true,3447 Solo Ave,Minor Damage,WV,Single Vehicle Collision,2000.00,1,NO,625.00,NO,6000.00,0,claims_1.json,2026-03-26T08:28:23.314Z,8625.00,null


In [0]:
# Calculate mean & stddev
stats = df.select(
    F.mean("total_amount").alias("mean"),
    F.stddev("total_amount").alias("std")
).collect()[0]

mean_val = stats["mean"]
std_val = stats["std"]

df = df.withColumn(
    "z_score",
    (F.col("total_amount") - mean_val) / std_val
)

df = df.withColumn(
    "rule_high_amount",
    F.when(F.col("z_score") > 2.5, 1).otherwise(0)
)

In [0]:
df = df.withColumn(
    "rule_severity_mismatch",
    F.when(
        (F.col("incident_severity").isin("Minor Damage", "Trivial Damage")) &
        (F.col("total_amount") > mean_val),
        1
    ).otherwise(0)
)

In [0]:
df = df.withColumn(
    "rule_missing_docs",
    F.when(
        (F.col("police_report_available").isin("NO", "?")) |
        (F.col("property_damage") == "?"),
        1
    ).otherwise(0)
)

In [0]:
window_policy = Window.partitionBy("Policy_id")

df = df.withColumn(
    "claim_count",
    F.count("*").over(window_policy)
)

df = df.withColumn(
    "rule_repeat_claimant",
    F.when(F.col("claim_count") > 2, 1).otherwise(0)
)

In [0]:
df = df.withColumn(
    "rule_delay",
    F.when(
        (F.col("processing_delay") > 30) |
        (F.col("Claim_Processed_On").isNull()),
        1
    ).otherwise(0)
)

In [0]:
df = df.withColumn(
    "anomaly_score",
    (F.col("rule_high_amount") * 30) +
    (F.col("rule_severity_mismatch") * 20) +
    (F.col("rule_missing_docs") * 15) +
    (F.col("rule_repeat_claimant") * 15) +
    (F.col("rule_delay") * 20)
)

In [0]:
df = df.withColumn(
    "priority",
    F.when(F.col("anomaly_score") >= 80, "HIGH")
     .when(F.col("anomaly_score") >= 60, "MEDIUM")
     .otherwise("LOW")
)

In [0]:
df = df.withColumn(
    "triggered_rules",
    F.concat_ws(", ",
        F.when(F.col("rule_high_amount") == 1, "High Amount"),
        F.when(F.col("rule_severity_mismatch") == 1, "Severity Mismatch"),
        F.when(F.col("rule_missing_docs") == 1, "Missing Docs"),
        F.when(F.col("rule_repeat_claimant") == 1, "Repeat Claimant"),
        F.when(F.col("rule_delay") == 1, "Processing Delay")
    )
)

In [0]:
anomalies_df = df.filter(F.col("anomaly_score") >= 60)

display(anomalies_df)

claim_id,Claim_Logged_On,_raw_logged_on,claim_logged_on_corrupted,Claim_Processed_On,_raw_processed_on,claim_processed_on_corrupted,claim_rejected,policy_id,authorities_contacted,bodily_injuries,collision_type,incident_city,incident_date,_raw_incident_date,incident_date_corrupted,incident_location,incident_severity,incident_state,incident_type,injury,number_of_vehicles_involved,police_report_available,property,property_damage,vehicle,witnesses,source_file,ingestion_timestamp,total_amount,processing_delay,z_score,rule_high_amount,rule_severity_mismatch,rule_missing_docs,claim_count,rule_repeat_claimant,rule_delay,anomaly_score,priority,triggered_rules
17284302,null,50:00.0,true,null,23:00.0,true,N,183430,Police,1,Rear Collision,Northbrook,null,31:00.0,true,7477 MLK Drive,Minor Damage,NY,Multi-vehicle Collision,51500.00,3,null,280.00,NO,900.00,0,claims_2.json,2026-03-26T08:28:24.955Z,52680.00,null,3.6814561053524733,1,1,0,1,0,1,70,MEDIUM,"High Amount, Severity Mismatch, Processing Delay"
17582682,null,39:00.0,true,null,43:00.0,true,N,235220,Other,1,Rear Collision,Northbrook,null,32:00.0,true,1512 Rock Lane,Minor Damage,WV,Multi-vehicle Collision,55000.00,4,YES,93.15,YES,4599.99,2,claims_2.json,2026-03-26T08:28:24.955Z,59693.14,null,4.329760852579048,1,1,0,1,0,1,70,MEDIUM,"High Amount, Severity Mismatch, Processing Delay"
7300704,null,44:00.0,true,null,06:00.0,true,N,253791,Ambulance,2,Front Collision,Arlington,null,20:00.0,true,8907 Tree Ave,Major Damage,WV,Single Vehicle Collision,14500.00,1,NO,5150.00,NO,26000.00,1,claims_1.json,2026-03-26T08:28:23.314Z,45650.00,null,3.031592795479461,1,0,1,1,0,1,65,MEDIUM,"High Amount, Missing Docs, Processing Delay"
17697389,null,19:00.0,true,null,NULL,false,Y,260845,Other,0,Front Collision,Riverwood,null,44:00.0,true,9603 Texas Lane,Minor Damage,WV,Single Vehicle Collision,11000.00,1,NO,550.00,NO,54000.00,2,claims_2.json,2026-03-26T08:28:24.955Z,65550.00,null,4.871177413754418,1,1,1,1,0,1,85,HIGH,"High Amount, Severity Mismatch, Missing Docs, Processing Delay"
17294642,null,09:00.0,true,null,53:00.0,true,N,263159,Ambulance,0,Rear Collision,Northbend,null,33:00.0,true,7756 Solo Drive,Minor Damage,SC,Single Vehicle Collision,41000.00,1,null,480.00,YES,1300.00,1,claims_2.json,2026-03-26T08:28:24.955Z,42780.00,null,2.7662858681202085,1,1,0,1,0,1,70,MEDIUM,"High Amount, Severity Mismatch, Processing Delay"
17100547,null,28:00.0,true,null,28:00.0,true,N,291902,Ambulance,1,Side Collision,Arlington,null,54:00.0,true,3982 Weaver Lane,Minor Damage,NY,Multi-vehicle Collision,51500.00,3,null,210.00,YES,6500.00,3,claims_1.json,2026-03-26T08:28:23.314Z,58210.00,null,4.192657258069081,1,1,0,1,0,1,70,MEDIUM,"High Amount, Severity Mismatch, Processing Delay"
18253896,null,23:00.0,true,null,56:00.0,true,N,309323,Police,0,null,Arlington,null,48:00.0,true,7609 Rock St,Trivial Damage,NC,Parked Car,15000.00,1,NO,450.00,NO,38000.00,3,claims_2.json,2026-03-26T08:28:24.955Z,53450.00,null,3.7526360126927605,1,1,1,1,0,1,85,HIGH,"High Amount, Severity Mismatch, Missing Docs, Processing Delay"
17697384,null,47:00.0,true,null,01:00.0,true,N,328387,Police,2,null,Hillsdale,null,58:00.0,true,2725 Britain Ridge,Trivial Damage,SC,Parked Car,7000.00,1,NO,1200.00,YES,51500.00,2,claims_2.json,2026-03-26T08:28:24.955Z,59700.00,null,4.330395000844443,1,1,1,1,0,1,85,HIGH,"High Amount, Severity Mismatch, Missing Docs, Processing Delay"
18235425,null,33:00.0,true,null,NULL,false,N,348209,None,0,null,Arlington,null,33:00.0,true,9918 Andromedia Drive,Minor Damage,WV,Parked Car,4000.00,1,null,550.00,NO,51500.00,1,claims_2.json,2026-03-26T08:28:24.955Z,56050.00,null,3.9929837517638602,1,1,0,1,0,1,70,MEDIUM,"High Amount, Severity Mismatch, Processing Delay"
208939,null,25:00.0,true,null,33:00.0,true,N,379268,Ambulance,1,Rear Collision,Arlington,null,57:00.0,true,6603 Francis Hwy,Minor Damage,WV,Single Vehicle Collision,6550.00,1,YES,600.00,YES,54000.00,0,claims_1.json,2026-03-26T08:28:23.314Z,61150.00,null,4.464435086095633,1,1,0,1,0,1,7

In [0]:
def build_prompt(row):
    return f"""
    You are a fraud investigator.

    Claim Details:
    - Claim Amount: {row['total_amount']}
    - Severity: {row['incident_severity']}
    - Incident Type: {row['incident_type']}
    - Delay: {row['processing_delay']}
    - Rules Triggered: {row['triggered_rules']}

    Provide:
    1. Why this claim is suspicious (specific data points)
    2. Risk factors
    3. Recommended next steps
    """

# Convert to Pandas (small dataset assumption)
pdf = anomalies_df.toPandas()

# Mock LLM call (replace with real API)
def generate_brief(row):
    return f"""
    Claim flagged due to {row['triggered_rules']}.
    High risk indicators observed in claim amount and documentation.
    Recommended: Verify documents, contact witnesses, audit claim history.
    """

pdf["investigation_brief"] = pdf.apply(generate_brief, axis=1)

final_df = spark.createDataFrame(pdf)

In [0]:
final_df.select(
    "Claim_ID",
    "Policy_ID",
    "total_amount",
    "incident_severity",
    "incident_type",
    "anomaly_score",
    "priority",
    "triggered_rules",
    "investigation_brief"
).write.mode("overwrite").saveAsTable("primeinsurance.gold_layer.claim_anomaly_explanations")

In [0]:
spark.sql("SELECT COUNT(*) FROM primeinsurance.gold_layer.claim_anomaly_explanations").show()

display(spark.table("primeinsurance.gold_layer.claim_anomaly_explanations"))

+--------+
|COUNT(*)|
+--------+
|      35|
+--------+



Claim_ID,Policy_ID,total_amount,incident_severity,incident_type,anomaly_score,priority,triggered_rules,investigation_brief
17284302,183430,52680.00,Minor Damage,Multi-vehicle Collision,70,MEDIUM,"High Amount, Severity Mismatch, Processing Delay","Claim flagged due to High Amount, Severity Mismatch, Processing Delay. High risk indicators observed in claim amount and documentation. Recommended: Verify documents, contact witnesses, audit claim history."
17582682,235220,59693.14,Minor Damage,Multi-vehicle Collision,70,MEDIUM,"High Amount, Severity Mismatch, Processing Delay","Claim flagged due to High Amount, Severity Mismatch, Processing Delay. High risk indicators observed in claim amount and documentation. Recommended: Verify documents, contact witnesses, audit claim history."
7300704,253791,45650.00,Major Damage,Single Vehicle Collision,65,MEDIUM,"High Amount, Missing Docs, Processing Delay","Claim flagged due to High Amount, Missing Docs, Processing Delay. High risk indicators observed in claim amount and documentation. Recommended: Verify documents, contact witnesses, audit claim history."
17697389,260845,65550.00,Minor Damage,Single Vehicle Collision,85,HIGH,"High Amount, Severity Mismatch, Missing Docs, Processing Delay","Claim flagged due to High Amount, Severity Mismatch, Missing Docs, Processing Delay. High risk indicators observed in claim amount and documentation. Recommended: Verify documents, contact witnesses, audit claim history."
17294642,263159,42780.00,Minor Damage,Single Vehicle Collision,70,MEDIUM,"High Amount, Severity Mismatch, Processing Delay","Claim flagged due to High Amount, Severity Mismatch, Processing Delay. High risk indicators observed in claim amount and documentation. Recommended: Verify documents, contact witnesses, audit claim history."
17100547,291902,58210.00,Minor Damage,Multi-vehicle Collision,70,MEDIUM,"High Amount, Severity Mismatch, Processing Delay","Claim flagged due to High Amount, Severity Mismatch, Processing Delay. High risk indicators observed in claim amount and documentation. Recommended: Verify documents, contact witnesses, audit claim history."
18253896,309323,53450.00,Trivial Damage,Parked Car,85,HIGH,"High Amount, Severity Mismatch, Missing Docs, Processing Delay","Claim flagged due to High Amount, Severity Mismatch, Missing Docs, Processing Delay. High risk indicators observed in claim amount and documentation. Recommended: Verify documents, contact witnesses, audit claim history."
17697384,328387,59700.00,Trivial Damage,Parked Car,85,HIGH,"High Amount, Severity Mismatch, Missing Docs, Processing Delay","Claim flagged due to High Amount, Severity Mismatch, Missing Docs, Processing Delay. High risk indicators observed in claim amount and documentation. Recommended: Verify documents, contact witnesses, audit claim history."
18235425,348209,56050.00,Minor Damage,Parked Car,70,MEDIUM,"High Amount, Severity Mismatch, Processing Delay","Claim flagged due to High Amount, Severity Mismatch, Processing Delay. High risk indicators observed in claim amount and documentation. Recommended: Verify documents, contact witnesses, audit claim history."
208939,379268,61150.00,Minor Damage,Single Vehicle Collision,70,MEDIUM,"High Amount, Severity Mismatch, Processing Delay","Claim flagged due to High Amount, Severity Mismatch, Processing Delay. High risk indicators observed in claim amount and documentation. Recommended: Verify documents, contact witnesses, audit claim history."
